# Library

In [1]:
import socket
import binascii
import time

import threading
import logging

import zmq

In [2]:
%run iota_global.ipynb

In [3]:
%run iota_im_dbwriter.ipynb

# Initialization

## Logger

In [4]:
logging.basicConfig(
    filename=PATH_SERVER_LOG,
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s:%(message)s",
)

iota_srv_logger = logging.getLogger(__name__)    

def log_event(aEvent, aMsg):
    return iota_srv_logger.info(f'{aEvent} |> {aMsg}')

def log_server_event(aMsg):
    log_event('Server event', aMsg)    

curr_msg = f'IoT server is loggoing in {PATH_SERVER_LOG}'    
log_server_event(curr_msg)    

## DBWriter

In [5]:
dbwriter_type, dbwriter_dev = DBWRITER_TYPE_SQLITE, DB_PATH_DEVICE_SQLITE

iota_srv_dbwriter = get_dbwriter(dbwriter_type, dbwriter_dev)
assert iota_srv_dbwriter is not None

curr_msg = f'IoT server data writer connected to {dbwriter_type}@{dbwriter_dev}'    
log_server_event(curr_msg)

## Data Publisher

In [6]:
iota_srv_pub_context = zmq.Context()
iota_srv_publisher = iota_srv_pub_context.socket(zmq.PUB)
iota_srv_publisher.bind(SOCK_TCP_MCUSRV_PUB)

curr_msg = f'IoT server is publishing data at {SOCK_TCP_MCUSRV_PUB}'    
log_server_event(curr_msg)

## Network socket

In [7]:
iota_srv_net_daemon = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
iota_srv_net_daemon.bind((IOTA_SVR_ADDR, IOTA_SVR_PORT_LTN))
iota_srv_net_daemon.settimeout(IN_SOCK_TIMEOUT_SEC) 

curr_msg = f'IoT server is listening on {IOTA_SVR_ADDR}:{str(IOTA_SVR_PORT_LTN)}'    
log_server_event(curr_msg)
print(curr_msg) 

IoT server is listening on 0.0.0.0:33333


# Need to manuly start the realtime analytical subscribers

# IoT server

## Datagram

- Header
1. UId: Device unit ID
2. PkgS: Package serial for UDP drop detection
3. GType: Datagram type, fixed as 'DM' for now
4. Tick: clock tick on device
- Data 
5. JSON string
- Example: UId:TestDeviceMPU|PkgS:13|GType:DM|Tick:168888|{"AccX":-3.36, "AccY":-6.19, "AccZ":5.18}

In [8]:
PROTOCOL_SECTION_CT = 5
PROTOCOL_SECTION_SEP = '|'

DG_TYPE_UNKNOWN = 'U'

DG_TYPE_DATA = 'MPU'
DG_TYPE_SET = {DG_TYPE_DATA}

In [9]:
def parse_datagram(aDg):
    pkg_pieces = aDg.split(PROTOCOL_SECTION_SEP)    
    if len(pkg_pieces)==PROTOCOL_SECTION_CT:
        dgt = pkg_pieces[2].split(':')
        dgt = dgt[-1].strip()   
        if dgt in DG_TYPE_SET:
            uid = pkg_pieces[0].split(':')[1]
            bid = int(pkg_pieces[1].split(':')[1])
            tick = int(pkg_pieces[3].split(':')[1])
            body = pkg_pieces[4].strip()
            return {'type':dgt, 'uid':uid, 'bid':bid, 'tick':tick, 'body':body}
            
    return {'type':DG_TYPE_UNKNOWN, 'uid':0, 'bid':0, 'tick':0, 'body':aDg}  

## Prepare Infinit loop

In [10]:
IOTA_SRV_EVENT_EXIT = threading.Event()

def run_iot_srv_loop():
    while not IOTA_SRV_EVENT_EXIT.is_set():
        try:
            payload, client_address = iota_srv_net_daemon.recvfrom(IN_SOCK_BUF_SIZE)    
            dg = payload.decode()
            
            dg_sec = parse_datagram(dg)
            dg_type = dg_sec['type']
            if dg_type in DG_TYPE_DATA: 
                data = dg_sec['body']
                iota_srv_dbwriter.ingest(aUId=dg_sec['uid'], 
                                         aBId=dg_sec['bid'], 
                                         aTick=dg_sec['tick'], 
                                         aType=dg_type,
                                         aData=data)            
                iota_srv_publisher.send_string(f'{dg_type}{{"tick": {dg_sec["tick"]}, "metrics": {data} }}')
                log_event('Normal datagram', f'size={len(data)}')
            else:
                log_event('Unknow datagram', dg) 
        except socket.timeout:
            pass
    # END: while True

IOTA_SRV_THREAD = threading.Thread(target=run_iot_srv_loop)

## Start server

In [11]:
IOTA_SRV_THREAD.start()

curr_msg = f'IoT server started ...'    
log_server_event(curr_msg)
print(curr_msg) 

IoT server started ...


## Shutdown Server

# Need to shutdwon real-time subscribers manaully